GPT-2 reads each row as text with the label first, scores it as True and False, and uses Bayes' rule to predict fraud.
**CV 0.9500** on 5-fold stratified CV, `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`; this notebook runs in under a minute on a Kaggle CPU because it loads the saved predictions (training took about 60 GPU hours; see below).
Run it alone to create a submission, or attach its output to the [stacking notebook](https://www.kaggle.com/code/masayakawamata/s6e11-logreg-stacking-same-split-oofs).

The GPT-2 classifier method is [Chris Deotte](https://www.kaggle.com/cdeotte)'s: [1st Place - TFMs and LLMs with tabular data!](https://www.kaggle.com/competitions/playground-series-s6e10/discussion/754683) (S6E10). This applies that recipe to S6E11; any mistakes are mine.

The post also links [Stacking TFMs with Public OOF](https://www.kaggle.com/code/cdeotte/stacking-tfms-with-public-oof). The replica step uses GReaT: Borisov et al., [Language Models are Realistic Tabular Data Generators](https://arxiv.org/abs/2210.06280) (ICLR 2023), and the [be_great library](https://github.com/tabularis-ai/be_great).

The replica was trained on [Credit Card Fraud Detection 2026](https://www.kaggle.com/datasets/uditjain13/credit-card-fraud-detection-2026) by [uditjain13](https://www.kaggle.com/uditjain13).

Series index: a Discussion post in this competition (link added when it is up).

[Training code](https://github.com/Matapanino/s6e11-gpt2-target-first) · [Prediction dataset](https://www.kaggle.com/datasets/masayakawamata/s6e11-gpt2-target-first-oof)

In [1]:
# Choose whether to show the text examples; the trained predictions are attached.
SHOW_ROW_TEXT = True

In [2]:
# Load the CPU libraries and choose the input and output folders.
import csv
import time
from IPython.display import display
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

started = time.perf_counter()
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working")

In [3]:
# Find saved predictions and keep their split, scale and CSV row order.
import hashlib
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

SPLIT = 'StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(np.zeros(n), y)'


def row_to_text(row, columns, label, order=None):
    """Pure Python copy of the target-first text rule; values remain CSV strings."""
    if label not in ('True', 'False'):
        raise ValueError('label must be True or False')
    if order is None:
        order = range(len(columns))
    fields = []
    for j in order:
        value = row[columns[j]].strip() or 'None'
        fields.append(f'{columns[j]} is {value}')
    return f'is_fraud is {label}, ' + ', '.join(fields) + '<|endoftext|>'


def example_rows(train_csv):
    import csv
    with Path(train_csv).open(newline='') as f:
        reader = csv.DictReader(f)
        first = next(reader)
        positive = next(row for row in reader if row['is_fraud'].strip() in ('1', '1.0', 'True'))
    return [first, positive]


def find_one(root, filename):
    paths = sorted(p for p in Path(root).rglob(filename) if p.is_file())
    if not paths:
        raise FileNotFoundError(f'{filename} was not found beneath {root}')
    if len(paths) != 1:
        raise ValueError(f'Expected one {filename}; found {paths}')
    return paths[0]


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def rebuild_folds(y):
    fold_ids = np.full(len(y), -1, dtype=np.int8)
    for fold, (_, held_out) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(np.zeros(len(y)), y)):
        fold_ids[held_out] = fold
    return fold_ids


def sigmoid(values):
    values = np.asarray(values, dtype=np.float64)
    result = np.empty_like(values)
    positive = values >= 0
    result[positive] = 1.0 / (1.0 + np.exp(-values[positive]))
    exp_values = np.exp(values[~positive])
    result[~positive] = exp_values / (1.0 + exp_values)
    return result


def load_member(input_root, verbose=True):
    names = ['train.csv', 'test.csv', 'oof.npy', 'test.npy', 'fold_ids.npy', 'manifest.json']
    paths = {name: find_one(input_root, name) for name in names}
    if verbose:
        for name, path in paths.items():
            print(f'{name}: {path}')
    source = json.loads(paths['manifest.json'].read_text())
    if source['model'] != 'gpt2_target_first' or not source['output'].startswith('log-odds'):
        raise ValueError('Unexpected model or prediction scale')
    for name in ('oof.npy', 'test.npy'):
        if sha256(paths[name]) != source['sha256'][name]:
            raise ValueError(f'{name} checksum differs')
    for key, name in [('train_csv', 'train.csv'), ('test_csv', 'test.csv'), ('fold_ids', 'fold_ids.npy')]:
        if sha256(paths[name]) != source['inputs'][key]['sha256']:
            raise ValueError(f'{name} checksum differs; CSV row order must match')
    train, test = pd.read_csv(paths['train.csv']), pd.read_csv(paths['test.csv'])
    if not train.id.is_unique or not test.id.is_unique:
        raise ValueError('IDs must be unique within each CSV')
    y = train.is_fraud.to_numpy(dtype=np.int8)
    if not np.array_equal(np.unique(y), [0, 1]):
        raise ValueError('Expected binary labels')
    folds = rebuild_folds(y)
    saved_folds = np.load(paths['fold_ids.npy'], allow_pickle=False)
    if saved_folds.dtype != np.int8 or not np.array_equal(folds, saved_folds):
        raise ValueError('Saved folds differ from the rebuilt split')
    arrays = {}
    for key, count in [('oof', len(train)), ('test_scores', len(test))]:
        name = 'oof.npy' if key == 'oof' else 'test.npy'
        values = np.load(paths[name], allow_pickle=False)
        if values.dtype != np.float64 or values.shape != (count,) or not np.isfinite(values).all():
            raise ValueError(f'{name} has the wrong shape, dtype or nonfinite values')
        arrays[key] = values
    auc = float(roc_auc_score(y, arrays['oof']))
    fold_auc = [float(roc_auc_score(y[folds == k], arrays['oof'][folds == k])) for k in range(5)]
    if abs(auc - source['auc']['pooled']) > 1e-12:
        raise ValueError('Pooled AUC differs from the source manifest')
    for entry in source['per_fold']:
        if abs(fold_auc[entry['fold']] - entry['auc']) > 1e-12:
            raise ValueError('Fold AUC differs from the source manifest')
    return dict(train=train, test=test, y=y, fold_ids=folds, cv_auc=auc, fold_auc=fold_auc,
                source=source, paths=paths, **arrays)


def write_member(member, output_root, runtime_minutes):
    out = Path(output_root)
    out.mkdir(parents=True, exist_ok=True)
    for filename, values in [('oof.npy', member['oof']), ('test.npy', member['test_scores']), ('fold_ids.npy', member['fold_ids'])]:
        np.save(out / filename, values, allow_pickle=False)
    for filename, frame, scores in [('oof.csv', member['train'], member['oof']), ('submission.csv', member['test'], member['test_scores'])]:
        pd.DataFrame({'id': frame.id.to_numpy(), 'is_fraud': sigmoid(scores)}).to_csv(out / filename, index=False)
    manifest = dict(name='gpt2_target_first', model='GPT-2 small target-first generative classifier',
                    cv_auc=member['cv_auc'], fold_auc=member['fold_auc'], split=SPLIT,
                    prediction_scale='log-odds', train_rows=len(member['train']), test_rows=len(member['test']),
                    runtime_minutes=float(runtime_minutes))
    (out / 'manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
    return manifest

In [4]:
# Load contextual results from the companion public runs.
presentation = {'credits': "The method is [Chris Deotte](https://www.kaggle.com/cdeotte)'s: [1st Place - TFMs and LLMs with tabular data!](https://www.kaggle.com/competitions/playground-series-s6e10/discussion/754683) (S6E10). This applies that recipe to S6E11; any mistakes are mine.\n\nThe post also links [Stacking TFMs with Public OOF](https://www.kaggle.com/code/cdeotte/stacking-tfms-with-public-oof). The replica step uses GReaT: Borisov et al., [Language Models are Realistic Tabular Data Generators](https://arxiv.org/abs/2210.06280) (ICLR 2023), and the [be_great library](https://github.com/tabularis-ai/be_great).\n\nThe replica was trained on [Credit Card Fraud Detection 2026](https://www.kaggle.com/datasets/uditjain13/credit-card-fraud-detection-2026) by [uditjain13](https://www.kaggle.com/uditjain13).\n", 'costs': {'replica_hours': 5.0735921476946935, 'classifier_hours': 55.216101157334116, 'training_hours': 60.28969330502881, 'scoring_hours': 61.61892313056522, 'assembly_minutes': 24.857678584257762}, 'other_members': [{'Model': 'LogReg + TE features', 'CV AUC, same 5 folds': 0.9502480981230691, 'Notebook': 'https://www.kaggle.com/code/masayakawamata/s6e11-single-logreg-te-features-cv-0-9502'}, {'Model': 'XGB + TE features', 'CV AUC, same 5 folds': 0.9504396424046064, 'Notebook': 'https://www.kaggle.com/code/masayakawamata/s6e11-single-xgb-te-features-cv-0-9504'}], 'fold0_read': {'auc_delta': 0.0046368953334697816, 'baseline_auc_same_rows': 0.9441617102979585, 'gpt2_auc': 0.9487986056314283, 'label_fit': {'negative_rows': {'mean_log_p_false': -122.08970231840615, 'mean_log_p_true': -124.210998448149, 'mean_true_minus_false': -2.1212961297428508, 'rows': 137655}, 'positive_rows': {'mean_log_p_false': -127.55514093747512, 'mean_log_p_true': -124.50009656575469, 'mean_true_minus_false': 3.055044371720414, 'rows': 2337}}, 'model_sha256': 'f324727aa90ce4a257d11af4d9494859d001f11ddf552e646ebc54293b7fc493', 'positives': 2337, 'prior': 0.016700954340248014, 'rows': 139992, 'score_chunks': 70, 'time_utc': '2026-11-02T20:36:18.883988+00:00'}}

In [5]:
# Display and save each section's result in call order.
"""Paste this entire file into one hidden notebook cell before the finding cells."""

from html import escape as _evidence_escape
from numbers import Number as _EvidenceNumber
from pathlib import Path as _EvidencePath
import re as _evidence_re

import pandas as _evidence_pd
from matplotlib.figure import Figure as _EvidenceFigure
from IPython.display import HTML as _EvidenceHTML, display as _evidence_display

_evidence_call_number = 0
_evidence_style = """<style>
.s6e11-evidence table {border-collapse:collapse;width:100%;font-size:14px;margin:12px 0}
.s6e11-evidence th,.s6e11-evidence td {border-bottom:1px solid #cdd6df;padding:6px 9px;text-align:left}
.s6e11-evidence th {border-top:1px solid #96a7b7}
.s6e11-evidence .evidence-number strong {font-size:36px;color:#164b79}
.s6e11-evidence .evidence-number p,.s6e11-evidence .evidence-caption {margin:3px 0;color:#536273}
</style>"""


def show_evidence(name, obj, caption=None):
    """Display one result and save its reader-facing copy in call order."""
    global _evidence_call_number
    if not isinstance(name, str) or not _evidence_re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]*", name):
        raise ValueError("name must use letters, digits, underscores or hyphens")
    if caption is not None and (not isinstance(caption, str) or "\n" in caption or "\r" in caption):
        raise ValueError("caption must be one line of text")
    if isinstance(obj, _evidence_pd.DataFrame):
        formatters = {}
        for column in obj.select_dtypes(include="number").columns:
            formatters[column] = lambda value: "" if _evidence_pd.isna(value) else (f"{value:,.0f}" if float(value).is_integer() else f"{value:,.4f}".rstrip("0").rstrip("."))
        table = obj.to_html(index=False, escape=True, border=0, classes="evidence-table", formatters=formatters)
        content = "<div class='s6e11-evidence'>" + _evidence_style + table + (f"<p class='evidence-caption'>{_evidence_escape(caption)}</p>" if caption else "") + "</div>"
        suffix = ".html"
    elif isinstance(obj, _EvidenceFigure):
        content = obj
        suffix = ".png"
    elif isinstance(obj, (_EvidenceNumber, str)) and not isinstance(obj, bool):
        if isinstance(obj, str) and (len(obj) > 160 or "\n" in obj or "\r" in obj):
            raise ValueError("string evidence must be short and one line")
        value = str(obj) if isinstance(obj, str) else (f"{obj:,}" if isinstance(obj, int) else f"{float(obj):,.4f}".rstrip("0").rstrip("."))
        content = "<div class='s6e11-evidence'>" + _evidence_style + f"<div class='evidence-number'><strong>{_evidence_escape(value)}</strong>"
        content += f"<p>{_evidence_escape(caption)}</p>" if caption else ""
        content += "</div></div>"
        suffix = ".html"
    else:
        raise TypeError("show_evidence accepts a pandas DataFrame, matplotlib Figure, number or short string")

    _evidence_call_number += 1
    directory = OUTPUT_ROOT / "evidence"
    directory.mkdir(parents=True, exist_ok=True)
    path = directory / f"{_evidence_call_number:02d}_{name}{suffix}"
    if suffix == ".png":
        caption_text = content.text(0.5, 0.005, caption, ha="center", fontsize=10) if caption else None
        try:
            content.savefig(path, dpi=160, bbox_inches="tight")
            _evidence_display(content)
        finally:
            if caption_text is not None:
                caption_text.remove()
    else:
        path.write_text(content, encoding="utf-8")
        _evidence_display(_EvidenceHTML(content))

In [6]:
# Find the inputs even when Kaggle puts files several folders deep.
member = load_member(INPUT_ROOT)
train, test = member['train'], member['test']
source_manifest = member['source']
print(f"{len(train):,} training rows; {len(test):,} test rows")

## How it works

The recipe learns to write rows, then asks which label makes a row more likely.

1. Train a replica of the row generator on the original dataset with GReaT.
2. Continue training on each fold's fit rows, writing the label first and shuffling the feature order.
3. Score each held-out row with True and False, averaging the row likelihoods across feature orders before applying Bayes' rule.

<div style="break-before: page; page-break-before: always"></div>

The recipe follows the post, with fewer feature orders for the shared predictions. The [code README](https://github.com/Matapanino/s6e11-gpt2-target-first#the-recipe-and-where-each-setting-comes-from) also lists the choices the post leaves open.

In [7]:
# Keep the post's settings separate from the scoring budget used here.
recipe = pd.DataFrame([
    ['Base model and tokenizer', 'GPT-2 small', 'Post'],
    ['Replica', 'GReaT, batch 16, 60 epochs', 'Post'],
    ['Classifier text and loss', 'Target first; shuffled features; loss on all tokens', 'Post'],
    ['Classifier training', '8 epochs; AdamW; lr 5e-5 to 0; batch 32; bf16', 'Post'],
    ['Row text', 'column is value; comma separator; missing None', 'Post'],
    ['Score', 'Sum feature log-probabilities, including end token', "Post's code"],
    ['Combine feature orders', 'Log-sum-exp per label, then difference and fit-fold prior', 'Post'],
    ['Feature orders in these predictions', '10', 'This S6E11 run; post uses 30'],
    ['Stack', 'Log-odds clipped at +/-30; L2, C=1; no scaling', 'Post and linked stacking notebook'],
], columns=['Setting', 'Value', 'Source'])
show_evidence('recipe', recipe)

The following two full row strings show the formatting in detail. The label shown is the observed label; prediction scores both labels.

In [8]:
# Show the exact strings used for two real rows, including the end marker.
example_strings = example_rows(member['paths']['train.csv'])
columns = [c for c in example_strings[0] if c not in ('id', 'is_fraud')]
row_examples = pd.DataFrame([
    {'Row ID': row['id'], 'Text read by GPT-2': row_to_text(
        row, columns, 'True' if row['is_fraud'].strip() in ('1', '1.0', 'True') else 'False')}
    for row in example_strings
])
if SHOW_ROW_TEXT:
    show_evidence('row_text', row_examples)

## How I tested it

- Each held-out row was scored by a fold model that did not train on that row.
- All comparisons use the same five stratified folds and one training seed.
- I compared feature-order counts using the same saved scores.
- CV guided the recipe choices, so these folds are not an untouched final test.

In [9]:
# Use the same folds as the other members in this series.
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Results

This text model is useful as another view of the same rows. All scores below use the five stratified folds stated above.

In [10]:
# Read per-fold AUC and pooled CV directly from the attached predictions.
fold_results = pd.DataFrame({'Scored rows': [f'Fold {k}' for k in range(len(member['fold_auc']))] + ['Pooled OOF'],
                             'AUC': member['fold_auc'] + [member['cv_auc']]})
fold_results['AUC'] = fold_results['AUC'].map(lambda value: f'{value:.4f}')
show_evidence('fold_auc', fold_results)

The larger scoring budgets improved CV slightly. The gain column shows the small difference hidden by rounded AUC; it has not been tested on a fresh holdout.

In [11]:
# Display the order-count comparison stored by the prediction assembly script.
order_auc = source_manifest['diagnostics']['auc_by_number_of_orders']
orders_result = pd.DataFrame([
    {'Feature orders': int(count), 'CV AUC, same 5 folds': auc}
    for count, auc in sorted(order_auc.items(), key=lambda item: int(item[0]))
])
orders_result['Gain vs previous budget'] = orders_result['CV AUC, same 5 folds'].diff().map(lambda value: '' if pd.isna(value) else f'{value:+.1e}')
orders_result['CV AUC, same 5 folds'] = orders_result['CV AUC, same 5 folds'].map(lambda value: f'{value:.4f}')
show_evidence('feature_orders', orders_result)

The single-model scores give context beside my two public members. A lower single-model CV can still help when its mistakes differ.

See the [stacking notebook](https://www.kaggle.com/code/masayakawamata/s6e11-logreg-stacking-same-split-oofs) for the measured effect of adding GPT-2.

In [12]:
# Compare the public members on the same split; their CVs come from their run manifests.
public_members = presentation['other_members']
comparison = pd.DataFrame([{'Model': row['Model'], 'CV AUC': row['CV AUC, same 5 folds']} for row in public_members]
                          + [{'Model': 'GPT-2 target first', 'CV AUC': member['cv_auc']}])
comparison['CV AUC'] = comparison['CV AUC'].map(lambda value: format(value, '.4f'))
show_evidence('public_members', comparison)

## What a person had to know

A language model learns how likely the row is given a label. Putting the label first lets every feature depend on it. Score both labels, then use Bayes' rule and the training fold's fraud rate to turn those likelihoods into a prediction; a classifier head is unnecessary. This is the method in [Chris Deotte's GPT-2 classifier post](https://www.kaggle.com/competitions/playground-series-s6e10/discussion/754683).

The decision here was whether to keep paying for the full run. Ask the AI to compare GPT-2 and an XGBoost on the raw columns on exactly the same held-out rows. The completed first-fold comparison below supported continuing; a tiny early chunk would have been too noisy. This earlier check used the post's larger scoring budget, so it differs from the shared prediction above.

In [13]:
# Show the earlier fold-0 decision using the same held-out rows and 30 orders.
first_fold = pd.DataFrame([
    ['XGBoost on raw columns', presentation['fold0_read']['baseline_auc_same_rows']],
    ['GPT-2, 30 feature orders', presentation['fold0_read']['gpt2_auc']],
], columns=['Model on fold 0', 'AUC, same held-out rows'])
first_fold['AUC, same held-out rows'] = first_fold['AUC, same held-out rows'].map(lambda value: format(value, '.4f'))
show_evidence('first_fold_decision', first_fold)

In the separate [stacking notebook](https://www.kaggle.com/code/masayakawamata/s6e11-logreg-stacking-same-split-oofs), GPT-2 gave the largest gain among my same-split additions. That is a separate measurement from the first-fold decision above.

Cost also changed the delivery: the run took about 60.3 GPU hours for training and 61.6 for scoring, so I trained outside Kaggle and share the predictions here. These are summed job times. Scoring includes extra OOF orders for the first two folds; interrupted work and setup may be missing. This CPU notebook loads the predictions and writes the submission.

## Try next

- Attach this output to the stacking notebook and test its addition on the same folds.
- If training again, compare the feature-order counts in the table before increasing the scoring budget.
- Try another training seed on a fresh holdout; that repeat has not been tested here.

## What to take away

Use these predictions directly for a submission or as another member of a same-split stack.
Keep the CSV row order and fold rule when reusing them.

## Appendix: files and reuse

Run every cell in order. This notebook does not retrain. The [repository](https://github.com/Matapanino/s6e11-gpt2-target-first) contains the full training recipe and CPU tests; the [dataset](https://www.kaggle.com/datasets/masayakawamata/s6e11-gpt2-target-first-oof) contains the exact saved predictions and their original manifest.

The output files are `oof.npy` and `test.npy` (float64 log-odds in competition CSV row order), `fold_ids.npy` (int8, rebuilt from the split and checked against the dataset), `oof.csv`, `submission.csv` (IDs and fraud probabilities), and `manifest.json` (model, split, scores and this notebook's runtime). Apply sigmoid to the arrays for probabilities. `submission.csv` is ready to submit. Attach the output to the stacking notebook as a Type A member; use the output's manifest rather than the dataset's assembly manifest.

Other public members: [LogReg + TE features](https://www.kaggle.com/code/masayakawamata/s6e11-single-logreg-te-features-cv-0-9502) · [XGB + TE features](https://www.kaggle.com/code/masayakawamata/s6e11-single-xgb-te-features-cv-0-9504).

The table of other members uses their published run manifests; this notebook calculates the GPT-2 scores from the attached arrays. The earlier first-fold check uses the completed raw-column baseline on the same rows. Training and scoring time comes from the training summaries and summed score-chunk times; the replica is included in training. CPU prediction assembly time is separate from GPU time.

Code is MIT licensed. The predictions are CC BY 4.0, with the credits above.

In [14]:
# Write the member contract and the submission in input CSV order.
output_manifest = write_member(member, OUTPUT_ROOT, (time.perf_counter() - started) / 60)
print(json.dumps(output_manifest, indent=2))